# Develop acceptance loop

In [ ]:
def accept_based_on_top_percentent_of_arbitrary_var(
        features : torch.Tensor, 
        default_flag : torch.Tensor, 
        var_for_rule : int,
        top_percent : float,
        default_value : int = 1, # 1 or 0
        min_count_bads : int = 4
):
    if var_for_rule >= features.shape[1]:
        raise ValueError("var_for_rule outside of index")
    
    cutoff = torch.quantile(features[:, var_for_rule], 1 - top_percent)

    accepts = features[:, var_for_rule] >= cutoff


    count_defaults_within_accepts = (default_flag[accepts] == default_value).sum()
    if count_defaults_within_accepts < min_count_bads:
        defaults_still_selectable = lidx_defaults_non_accepted.sum()
        if defaults_still_selectable == 0:
            return accepts
        
        count_bads_to_still_achieve = min_count_bads - count_defaults_within_accepts
        var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
        
        if defaults_still_selectable <= count_bads_to_still_achieve:
            var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
            accept_rule_to_include_all_defaults = features[:, var_for_rule] >=  var_for_rule_vals_of_rejected_defaults.min()
            return accept_rule_to_include_all_defaults
        
        new_cutoff = torch.topk(var_for_rule_vals_of_rejected_defaults,k=count_bads_to_still_achieve, largest=True).values[-1]

        return features[:, var_for_rule] >= new_cutoff
    
    return accepts
        


accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features,
    default_flag,
    var_for_rule=0,
    top_percent=top_percent
)
default_value = 1
min_count_bads : int = 4
var_for_rule : int = 0
count_defaults_within_accepts = (default_flag[accepts] == default_value).sum()

lidx_defaults_non_accepted = (~accepts) & (default_flag == default_value)
defaults_still_selectable = lidx_defaults_non_accepted.sum()


var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
new_cutoff = torch.topk(count_bads_to_still_achieve, k = 3, largest=True).values[-1]

In [ ]:
import torch

from credit_data_simulation import CreditDataGenerator, CreditData, accept_based_on_top_percentent_of_arbitrary_var

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 100
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0]),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]]),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]])
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    #dtype=torch.get_default_dtype()
)

# Initial population
data_gen.manual_seed(initial_seed)
features, default_flag = data_gen.sample(100)

accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features, 
    default_flag,
    var_for_rule=0,
    top_percent=top_percent,
    default_value = CreditDataGenerator.bad_good_encoding["bad"]
)

credit_data = CreditData(features, default_flag, accepts)

# Holdout Population
data_gen.manual_seed(-initial_seed)
holdout_features, holdout_flag = data_gen.sample(n=holdout_sample)
holdout_data = CreditData(
    holdout_features, holdout_flag, 
    accepted_initial=torch.ones(holdout_flag.shape, dtype=torch.bool) # All are "accepted"
)

# Acceptance Loop

stats = []

for gen_nr in range(1, num_gens + 1):
    if gen_nr % 10 == 0:
        print("-- Iteration", f"{gen_nr}/{num_gens}:", credit_data.accepted_count, 
              "accepts and", credit_data.rejected_count, " rejects")
        
    ## Gather current statistics
    current_stats : dict = credit_data.data_stats()


    data_gen.manual_seed(initial_seed + gen_nr)
    features, default_flag = data_gen.sample(100)

    




sel